# NB14 — Final PCA/LDA and calibration outputs

Public, output-stripped reproducibility notebook for the final manuscript. It does not retrain the benchmark models. It reads the original morphometric table and the saved OOF predictions from NB09.

**Recommended runtime:** CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import f1_score

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATASET = ROOT/'01_DATA'/'INIAP_Dataset.xlsx'
NB09 = ROOT/'03_RESULTS'/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
RAW_OOF = NB09/'all_oof_harmonized.csv'
CAL_OOF = NB09/'crossfit_temperature_scaled_oof.csv'
OUT = ROOT/'03_RESULTS'/'NB14_FINAL_PCA_LDA_CALIBRATION'
FIG = ROOT/'05_FIGURES'/'FINAL_PUBLICATION'
OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)

In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
TARGET='Class'
X=df.drop(columns=[TARGET])
y=df[TARGET].astype(str)
assert len(df)==4000 and y.nunique()==4

Xs=StandardScaler().fit_transform(X)
P=PCA(n_components=2).fit(Xs)
PC=P.transform(Xs)
le=LabelEncoder(); ye=le.fit_transform(y)
L=LinearDiscriminantAnalysis(n_components=2).fit(Xs,ye)
LD=L.transform(Xs)
classes=sorted(y.unique())

fig,ax=plt.subplots(1,2,figsize=(12.8,5.3))
for cls in classes:
    m=(y.values==cls); ax[0].scatter(PC[m,0],PC[m,1],s=17,alpha=.48,linewidths=0,label=cls)
ax[0].set_xlabel(f'PC1 ({P.explained_variance_ratio_[0]*100:.1f}% variance)')
ax[0].set_ylabel(f'PC2 ({P.explained_variance_ratio_[1]*100:.1f}% variance)')
ax[0].set_title('(a) Principal Component Analysis'); ax[0].grid(alpha=.15)
ax[1].set_prop_cycle(None)
for cls in classes:
    m=(y.values==cls); ax[1].scatter(LD[m,0],LD[m,1],s=17,alpha=.48,linewidths=0,label=cls)
ax[1].set_xlabel(f'LD1 ({L.explained_variance_ratio_[0]*100:.1f}% discriminant variance)')
ax[1].set_ylabel(f'LD2 ({L.explained_variance_ratio_[1]*100:.1f}% discriminant variance)')
ax[1].set_title('(b) Linear Discriminant Analysis'); ax[1].grid(alpha=.15)
h,l=ax[0].get_legend_handles_labels(); fig.legend(h,l,loc='lower center',ncol=4,frameon=False,bbox_to_anchor=(.5,-.01))
plt.tight_layout(rect=[0,.08,1,1])
stem=FIG/'Figure_3_PCA_LDA_INIAP_Dry_Bean'
plt.savefig(str(stem)+'.png',dpi=600,bbox_inches='tight'); plt.savefig(str(stem)+'.pdf',bbox_inches='tight'); plt.show()

pd.DataFrame({'PC1':PC[:,0],'PC2':PC[:,1],TARGET:y}).to_csv(OUT/'pca_scores_pc1_pc2.csv',index=False)
pd.DataFrame({'LD1':LD[:,0],'LD2':LD[:,1],TARGET:y}).to_csv(OUT/'lda_scores_ld1_ld2.csv',index=False)

In [ ]:
raw=pd.read_csv(RAW_OOF); cal=pd.read_csv(CAL_OOF)
prob_cols=[c for c in raw.columns if c.startswith('prob_')]
cal_prob_cols=['cal_'+c for c in prob_cols]
CLASSES=[c.replace('prob_','',1) for c in prob_cols]
N_BINS=10

def norm(P):
    P=np.clip(np.asarray(P,float),1e-12,None)
    return P/P.sum(axis=1,keepdims=True)

def reliability(ytrue,P):
    P=norm(P); cls=np.asarray(CLASSES); k=P.argmax(1); pred=cls[k]; conf=P[np.arange(len(P)),k]
    corr=(pred==np.asarray(ytrue).astype(str)).astype(int); edges=np.linspace(0,1,N_BINS+1); bid=np.digitize(conf,edges[1:-1],right=True)
    rows=[]; ece=0.
    for b in range(N_BINS):
        m=bid==b; n=int(m.sum())
        if not n: continue
        mc=float(conf[m].mean()); ac=float(corr[m].mean()); ece+=(n/len(conf))*abs(ac-mc)
        rows.append({'bin':b+1,'lower':edges[b],'upper':edges[b+1],'count':n,'mean_confidence':mc,'accuracy':ac,'absolute_gap':abs(ac-mc)})
    return pd.DataFrame(rows),ece

def brier(ytrue,P):
    P=norm(P); mp={c:i for i,c in enumerate(CLASSES)}; Y=np.zeros_like(P); yi=np.array([mp[str(v)] for v in ytrue]); Y[np.arange(len(yi)),yi]=1
    return float(np.mean(np.sum((P-Y)**2,axis=1)))

bins_all=[]; vals=[]
for model_id,label in [('TabPFN','TabPFN'),('LDA_XGBoost','LDA-XGBoost')]:
    for calname,dfp,pcols in [('Raw',raw,prob_cols),('Cross-fitted temperature',cal,cal_prob_cols)]:
        g=dfp[(dfp.feature_set=='FULL_11')&(dfp.model==model_id)]
        bt,e=reliability(g.y_true.astype(str).values,g[pcols].to_numpy()); bt['model_id']=model_id; bt['model_label']=label; bt['calibration']=calname; bins_all.append(bt)
    for seed in sorted(raw.seed.unique()):
        gr=raw[(raw.feature_set=='FULL_11')&(raw.model==model_id)&(raw.seed==seed)]
        gc=cal[(cal.feature_set=='FULL_11')&(cal.model==model_id)&(cal.seed==seed)]
        Pr=gr[prob_cols].to_numpy(); Pc=gc[cal_prob_cols].to_numpy()
        _,er=reliability(gr.y_true.astype(str).values,Pr); _,ec=reliability(gc.y_true.astype(str).values,Pc)
        pr=np.asarray(CLASSES)[norm(Pr).argmax(1)]; pc=np.asarray(CLASSES)[norm(Pc).argmax(1)]
        vals.append({'model':model_id,'model_label':label,'seed':int(seed),
                     'raw_macro_f1':f1_score(gr.y_true.astype(str),pr,average='macro'),'raw_brier':brier(gr.y_true,Pr),'raw_ece':er,
                     'temp_macro_f1':f1_score(gc.y_true.astype(str),pc,average='macro'),'temp_brier':brier(gc.y_true,Pc),'temp_ece':ec})
pd.concat(bins_all,ignore_index=True).to_csv(OUT/'figure7_reliability_bins.csv',index=False)
pd.DataFrame(vals).to_csv(OUT/'figure7_calibration_validation_by_seed.csv',index=False)
print('NB14 complete.')